In [ ]:
# @title
# 1. Clone repository
%cd /content
!rm -rf MobileMamba
!git clone https://github.com/lewandofskee/MobileMamba.git

# 2. Setup isolated Python 3.10 environment
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -p /content/mobilemamba_env python=3.10 -c conda-forge

### Download and Paste the mobilemamba_b4.pth, fpn.pth in "MobileMamba/weights/MobileMamba_B4/" and mobilemamba_s6.pth in "MobileMamba/weights/MobileMamba_S6/" and the evaluation scripts in "/"

In [ ]:
# @title
# 1. PyTorch 2.1.2 + CUDA 11.8 wheels
!/content/mobilemamba_env/bin/pip install \
    torch==2.1.2 \
    torchvision==0.16.2 \
    torchaudio==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu118

# 2. Pin numpy < 2.0 (ensures compatibility with torch 2.1 and cv2)
!/content/mobilemamba_env/bin/pip install "numpy<2"

# 3. Model dependencies and utilities matching repo specs
!/content/mobilemamba_env/bin/pip install \
    timm==0.9.16 \
    tensorboardX \
    einops \
    torchprofile \
    fvcore==0.1.5.post20221221 \
    triton==2.1.0 \
    lmdb \
    PyWavelets \
    scikit-image \
    six \
    "opencv-python==4.8.1.78" \
    terminaltables \
    pycocotools \
    prettytable \
    xtcocotools \
    mmpretrain==1.2.0 \
    mmdet==3.3.0 \
    mmsegmentation==1.2.2 \
    ftfy \
    regex \
    "setuptools<81" \
    mmcv==2.1.0 \
    -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html

In [ ]:
# @title
import os
import subprocess
import time
import urllib.request

url = "https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run"
dest = "/content/cuda_11.8.0.run"

# 1. Download with real-time progress bar
if not os.path.exists(dest):
    print("Downloading CUDA 11.8 runfile (~3.0 GB)...")
    def report_hook(count, block_size, total_size):
        downloaded = count * block_size
        pct = downloaded / total_size * 100
        mb_down = downloaded / (1024 * 1024)
        mb_total = total_size / (1024 * 1024)
        print(f"\rDownload Progress: [{pct:6.2f}%]  {mb_down:6.1f} MB / {mb_total:6.1f} MB", end="", flush=True)

    urllib.request.urlretrieve(url, dest, reporthook=report_hook)
    print("\nDownload complete!")
else:
    print("CUDA 11.8 runfile already present. Skipping download.")

os.chmod(dest, 0o755)

# 2. Silent Toolkit Install with live elapsed-time spinner
print("\nInstalling CUDA 11.8 Toolkit to /content/cuda-11.8 (takes ~2-3 mins)...")
proc = subprocess.Popen(
    ["sudo", dest, "--toolkit", "--silent", "--installpath=/content/cuda-11.8"]
)

spinner = ["|", "/", "-", "\\"]
start_time = time.time()
idx = 0

while proc.poll() is None:
    elapsed = int(time.time() - start_time)
    mins, secs = divmod(elapsed, 60)
    print(f"\rInstalling... {spinner[idx % len(spinner)]} [Elapsed Time: {mins:02d}:{secs:02d}]", end="", flush=True)
    idx += 1
    time.sleep(0.5)

if proc.returncode == 0:
    print(f"\nCUDA 11.8 installed successfully in {int(time.time() - start_time)} seconds!")
else:
    raise RuntimeError(f"\nCUDA installation failed with exit code: {proc.returncode}")

# 3. Create driver library symlink for CUDA runtime bindings
!mkdir -p /content/cuda-11.8/lib64
!sudo ln -sf /usr/lib64-nvidia/libcuda.so /content/cuda-11.8/lib64/libcuda.so

In [ ]:
# @title
# VERY IMPORTANT - INSTALL SELECTIVE SCAN OFLEX
import subprocess
import os

env_python = "/content/mobilemamba_env/bin/python"
root = "/content/MobileMamba/model/lib_mamba/kernels/selective_scan"

env = os.environ.copy()

result = subprocess.run(
    [
        env_python,
        "-m",
        "pip",
        "install",
        ".",
        "--no-build-isolation",
    ],
    cwd=root,
    env=env,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Author's selective-scan installation failed with code {result.returncode}"
    )

In [ ]:
# @title
# VERY IMPORTANT LIBCUDA FIX
!echo "/usr/lib64-nvidia" > /etc/ld.so.conf.d/nvidia.conf
!ldconfig

In [ ]:
# @title
!mkdir -p /content/imagenet
!wget -c "https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar" -P /content/imagenet/

In [ ]:
# @title
%%bash
cd /content/imagenet
mkdir -p val
tar -xf ILSVRC2012_img_val.tar -C val/
rm ILSVRC2012_img_val.tar
echo "Validation archive extracted."

In [ ]:
# @title
%%bash
cd /content/imagenet/val
wget -qO- https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh | bash
echo "ImageNet validation set ready at /content/imagenet/val"

In [ ]:
# @title
%cd /content/MobileMamba

# Change ImageNet loader to standard ImageFolder
!sed -i "s/data.type = 'ImageFolderLMDB'/data.type = 'DefaultCLS'/" configs/mobilemamba/mobilemamba_b4.py

# Point B4 config to our prepared ImageNet validation dataset
!sed -i "s|data.root = 'data/imagenet'|data.root = '/content/imagenet'|" configs/mobilemamba/mobilemamba_b4.py

print("B4 ImageNet config patched successfully.")

# **SMOKE TESTS**

In [ ]:
# @title
#Test 1
import subprocess

code = r'''
import os
import sys
import copy
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ============================================================================
# ENVIRONMENT
# ============================================================================

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = (
    "/content/MobileMamba/weights/"
    "MobileMamba_B4/mobilemamba_b4.pth"
)
IMAGENET_VAL = "/content/imagenet/val"

sys.path.insert(0, REPO)
sys.path.insert(
    0,
    "/content/MobileMamba/model/lib_mamba/kernels/selective_scan"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 90)
print("STEP 1: RAW BASELINE vs. CONV-BN FUSION + CUDNN BENCHMARK")
print("=" * 90)
print("Device     :", device)
print("Checkpoint :", CHECKPOINT_PATH)
print("ImageNet   :", IMAGENET_VAL)

# ============================================================================
# IMPORT MODEL & FUSION UTILITY
# ============================================================================

from model.mobilemamba.mobilemamba import (
    MobileMamba,
    CFG_MobileMamba_B4,
    replace_batchnorm,
)

# ============================================================================
# MODEL LOADER
# ============================================================================

def load_model():
    cfg = copy.deepcopy(CFG_MobileMamba_B4)
    model = MobileMamba(
        **cfg,
        num_classes=1000,
        distillation=False,
        forward_type="v052d",
    )

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        if "model" in checkpoint:
            state_dict = checkpoint["model"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    state_dict = {
        (k.replace("module.", "", 1) if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    return model

# ============================================================================
# SMOKE TEST (FAIL-FAST)
# ============================================================================

print("\nRunning fail-fast smoke test on fused architecture...")
smoke_model = load_model()
replace_batchnorm(smoke_model)
smoke_model = smoke_model.to(device).eval()

dummy_smoke = torch.randn(2, 3, 512, 512, device=device)
with torch.inference_mode():
    smoke_out = smoke_model(dummy_smoke)
    if isinstance(smoke_out, (tuple, list)):
        smoke_out = smoke_out[0]

assert smoke_out.shape == (2, 1000), f"Smoke test failed! Output shape: {smoke_out.shape}"
assert not torch.isnan(smoke_out).any(), "Smoke test failed! Output contains NaN."
del smoke_model, dummy_smoke, smoke_out
if device.type == "cuda":
    torch.cuda.empty_cache()
print("Smoke test: PASS (Fusion verified safe)\n")

# ============================================================================
# IMAGE DATASET (2048 IMAGES)
# ============================================================================

transform = transforms.Compose([
    transforms.Resize(585, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
subset = Subset(dataset, list(range(2048)))

accuracy_loader = DataLoader(
    subset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================================
# BENCHMARK STAGES (STAGE 1: LATENCY -> STAGE 2: ACCURACY)
# ============================================================================

@torch.inference_mode()
def measure_latency(model):
    batch_size = 32
    warmup = 30
    iterations = 100

    dummy = torch.randn(batch_size, 3, 512, 512, device=device)
    model.eval()

    # Warmup to stabilize clocks and compile kernels
    for _ in range(warmup):
        _ = model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        start_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]
        end_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]

        for i in range(iterations):
            start_events[i].record()
            _ = model(dummy)
            end_events[i].record()

        torch.cuda.synchronize()
        times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
    else:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            _ = model(dummy)
            end = time.perf_counter()
            times.append((end - start) * 1000.0)

    times.sort()
    mean_ms = sum(times) / len(times)
    median_ms = times[len(times) // 2]
    p95_ms = times[max(0, int(0.95 * len(times)) - 1)]
    fps = batch_size / (mean_ms / 1000.0)

    del dummy
    return {
        "mean_ms": mean_ms,
        "median_ms": median_ms,
        "p95_ms": p95_ms,
        "fps": fps,
    }

@torch.inference_mode()
def evaluate_accuracy(model, name):
    model.eval()
    correct1 = 0
    correct5 = 0
    total = 0

    for images, labels in accuracy_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[0]

        _, pred = output.topk(5, dim=1, largest=True, sorted=True)
        correct = (pred == labels.unsqueeze(1))

        correct1 += correct[:, 0].sum().item()
        correct5 += correct.any(dim=1).sum().item()
        total += labels.size(0)

    top1 = 100.0 * correct1 / total
    top5 = 100.0 * correct5 / total
    print(f"{name} -> Top-1: {top1:.3f}% ({correct1}/{total}) | Top-5: {top5:.3f}% ({correct5}/{total})")
    return top1, top5

# ============================================================================
# RUN CONFIGURATIONS
# ============================================================================

experiments = [
    ("RAW BASELINE", False, False),
    ("FUSED + CUDNN BENCHMARK", True, True),
]

results = {}

for name, do_fuse, do_benchmark in experiments:
    print("-" * 90)
    print(f"Testing: {name}")
    print("-" * 90)

    torch.backends.cudnn.benchmark = do_benchmark

    model = load_model()
    if do_fuse:
        replace_batchnorm(model)

    model = model.to(device).eval()

    # STAGE 1: Pure GPU Latency first (on cold GPU)
    latency = measure_latency(model)
    print(f"Mean Latency: {latency['mean_ms']:.3f} ms | FPS: {latency['fps']:.2f}")

    # STAGE 2: Accuracy check on subset
    top1, top5 = evaluate_accuracy(model, name)

    results[name] = {
        "top1": top1,
        "top5": top5,
        "latency": latency,
    }

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

# ============================================================================
# RESULTS COMPARISON
# ============================================================================

base = results["RAW BASELINE"]
opt = results["FUSED + CUDNN BENCHMARK"]

base_lat = base["latency"]["mean_ms"]
opt_lat = opt["latency"]["mean_ms"]
lat_diff = 100.0 * (opt_lat - base_lat) / base_lat

base_fps = base["latency"]["fps"]
opt_fps = opt["latency"]["fps"]
fps_diff = 100.0 * (opt_fps - base_fps) / base_fps

print("\n" + "=" * 90)
print("STEP 1 COMPARISON RESULTS")
print("=" * 90)
print(f"{'Metric':<25} {'Raw Baseline':>18} {'Fused Baseline':>18} {'Delta':>15}")
print("-" * 90)
print(f"{'Top-1 Accuracy':<25} {base['top1']:>17.3f}% {opt['top1']:>17.3f}% {opt['top1'] - base['top1']:>+14.3f}%")
print(f"{'Top-5 Accuracy':<25} {base['top5']:>17.3f}% {opt['top5']:>17.3f}% {opt['top5'] - base['top5']:>+14.3f}%")
print(f"{'Mean Latency':<25} {base_lat:>15.3f} ms {opt_lat:>15.3f} ms {lat_diff:>+14.2f}%")
print(f"{'Throughput (FPS)':<25} {base_fps:>18.2f} {opt_fps:>18.2f} {fps_diff:>+14.2f}%")
print("=" * 90)
'''

result = subprocess.run(
    [
        "/content/mobilemamba_env/bin/python",
        "-c",
        code
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)
print("\nRETURN CODE:", result.returncode)

In [ ]:
# @title
#Test 2
import subprocess

code = r'''
import os
import sys
import copy
import time
import torch
import torch.nn as nn

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = "/content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth"

sys.path.insert(0, REPO)
sys.path.insert(0, "/content/MobileMamba/model/lib_mamba/kernels/selective_scan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B4

cfg = copy.deepcopy(CFG_MobileMamba_B4)
model = MobileMamba(**cfg, num_classes=1000, distillation=False, forward_type="v052d")

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()

print("=" * 80)
print("MOBILEMAMBA-B4 EXECUTION TIME BREAKDOWN (BATCH SIZE = 32, 512x512)")
print("=" * 80)

dummy = torch.randn(32, 3, 512, 512, device=device)

# Global warmup
with torch.inference_mode():
    for _ in range(20):
        _ = model(dummy)
torch.cuda.synchronize()

@torch.inference_mode()
def time_module(func, inp, runs=50, warmup=10):
    for _ in range(warmup):
        out = func(inp)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(runs):
        out = func(inp)
    end.record()
    torch.cuda.synchronize()
    return (start.elapsed_time(end) / runs), out

with torch.inference_mode():
    # 1. Stem (Downsamples to 128x128)
    t_stem, x1 = time_module(model.patch_embed, dummy)

    # 2. Stage 1 (blocks1)
    t_b1, x2 = time_module(model.blocks1, x1)

    # 3. Stage 2 Total (blocks2)
    t_b2_total, x3 = time_module(model.blocks2, x2)

    # --- Stage 2 Sub-components Breakdown ---
    # Downsampling sequence: blocks2[0..2] performs downsample & channel expansion (200 -> 376)
    def run_stage2_downsample(x):
        return model.blocks2[2](model.blocks2[1](model.blocks2[0](x)))

    t_b2_down, x_down = time_module(run_stage2_downsample, x2)

    # Inspect a standard Mamba block in Stage 2 (block 3)
    b2_blk = model.blocks2[3]

    # Pre-mixer branch: dw0 and ffn0 already apply residual connections internally
    def run_pre_mixer(x):
        return b2_blk.dw0(b2_blk.ffn0(x))

    t_pre_mix, x_mid = time_module(run_pre_mixer, x_down)

    # Mixer Total
    t_mixer, x_post_mix = time_module(b2_blk.mixer, x_mid)

    # Mixer Internal: Global Op vs Local Op
    mixer_attn = b2_blk.mixer.m.attn
    global_in = x_mid[:, :mixer_attn.global_channels].contiguous()
    local_in = x_mid[:, mixer_attn.global_channels:mixer_attn.global_channels + mixer_attn.local_channels].contiguous()

    t_global_op, _ = time_module(mixer_attn.global_op, global_in)
    t_local_op, _ = time_module(mixer_attn.local_op, local_in)

    # Post-mixer branch: dw1 and ffn1 already apply residual connections internally
    def run_post_mixer(x):
        return b2_blk.dw1(b2_blk.ffn1(x))

    t_post_mix, _ = time_module(run_post_mixer, x_post_mix)

    # 4. Stage 3 (blocks3)
    t_b3, x4 = time_module(model.blocks3, x3)

    # 5. Head
    t_head, _ = time_module(lambda x: model.head(torch.nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)), x4)

t_total = t_stem + t_b1 + t_b2_total + t_b3 + t_head

print(f"{'Component':<40} {'Latency (ms)':>15} {'% of Total':>15}")
print("-" * 80)
print(f"{'1. Patch Embed (Stem, 128x128)':<40} {t_stem:>13.2f} ms {100*t_stem/t_total:>14.1f}%")
print(f"{'2. Stage 1 (blocks1, 128x128)':<40} {t_b1:>13.2f} ms {100*t_b1/t_total:>14.1f}%")
print(f"{'3. Stage 2 (blocks2, 64x64) Total':<40} {t_b2_total:>13.2f} ms {100*t_b2_total/t_total:>14.1f}%")
print(f"{'   - Downsampling Layer (blocks2[0..2])':<40} {t_b2_down:>13.2f} ms {100*t_b2_down/t_total:>14.1f}%")
print(f"{'   - Single Block Pre-Mixer (FFN0+DW0)':<40} {t_pre_mix:>13.2f} ms {100*t_pre_mix/t_total:>14.1f}%")
print(f"{'   - Single Block Mixer Total':<40} {t_mixer:>13.2f} ms {100*t_mixer/t_total:>14.1f}%")
print(f"{'     * Global Op (MBWTConv2d / Scan)':<40} {t_global_op:>13.2f} ms {100*t_global_op/t_total:>14.1f}%")
print(f"{'     * Local Op (DWConv)':<40} {t_local_op:>13.2f} ms {100*t_local_op/t_total:>14.1f}%")
print(f"{'   - Single Block Post-Mixer (FFN1+DW1)':<40} {t_post_mix:>13.2f} ms {100*t_post_mix/t_total:>14.1f}%")
print(f"{'4. Stage 3 (blocks3, 32x32)':<40} {t_b3:>13.2f} ms {100*t_b3/t_total:>14.1f}%")
print(f"{'5. Head & Pooling':<40} {t_head:>13.2f} ms {100*t_head/t_total:>14.1f}%")
print("-" * 80)
print(f"{'Calculated Total Sum':<40} {t_total:>13.2f} ms {'100.0%':>15}")
print("=" * 80)
'''

result = subprocess.run(["/content/mobilemamba_env/bin/python", "-c", code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
# @title
#Test 3
import subprocess

code = r'''
import os
import sys
import copy
import time
import torch
import torch.nn as nn

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = "/content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth"

sys.path.insert(0, REPO)
sys.path.insert(0, "/content/MobileMamba/model/lib_mamba/kernels/selective_scan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B4

cfg = copy.deepcopy(CFG_MobileMamba_B4)
model = MobileMamba(**cfg, num_classes=1000, distillation=False, forward_type="v052d")

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()

print("=" * 80)
print("PROBING MBWTConv2d: WAVELET BRANCH vs. MAMBA SCAN (512x512 INPUT)")
print("=" * 80)

@torch.inference_mode()
def time_op(func, inp, runs=100, warmup=15):
    for _ in range(warmup):
        _ = func(inp)
    torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    for _ in range(runs):
        _ = func(inp)
    end_event.record()
    torch.cuda.synchronize()
    return start_event.elapsed_time(end_event) / runs

with torch.inference_mode():
    dummy = torch.randn(32, 3, 512, 512, device=device)

    # --- Extract Stage 1 probe tensor ---
    # Stem downsamples to 32x32 for Stage 1 blocks
    feat_s1 = model.patch_embed(dummy)
    b1_blk = model.blocks1[0]
    b1_op = b1_blk.mixer.m.attn.global_op

    # Forward through pre-mixer modules to match exact input to mixer
    x_s1_pre = b1_blk.dw0(b1_blk.ffn0(feat_s1))
    x_s1 = x_s1_pre[:, :b1_op.in_channels].contiguous()

    # --- Extract Stage 2 probe tensor ---
    # Full Stage 1 execution:
    out_s1 = model.blocks1(feat_s1)

    # Stage 2 downsampling & channel expansion via blocks2[0..2] (200 -> 376 channels)
    down_s2 = model.blocks2[2](model.blocks2[1](model.blocks2[0](out_s1)))

    # Target Mamba block in Stage 2 (blocks2[3])
    b2_blk = model.blocks2[3]
    b2_op = b2_blk.mixer.m.attn.global_op

    # Forward through pre-mixer modules to match exact input to mixer
    x_s2_pre = b2_blk.dw0(b2_blk.ffn0(down_s2))
    x_s2 = x_s2_pre[:, :b2_op.in_channels].contiguous()

    del dummy, feat_s1, out_s1, down_s2, x_s1_pre, x_s2_pre
    torch.cuda.empty_cache()

    probes = [
        (f"Stage 1 ({x_s1.shape[2]}x{x_s1.shape[3]}, C={b1_op.in_channels})", b1_op, x_s1),
        (f"Stage 2 ({x_s2.shape[2]}x{x_s2.shape[3]}, C={b2_op.in_channels})", b2_op, x_s2)
    ]

    for stage_name, op, inp in probes:
        # 1. Total global_op (Wavelet + Mamba scan combined)
        t_total = time_op(op, inp)

        # 2. Only Mamba SS2D scan branch
        t_mamba = time_op(lambda x: op.base_scale(op.global_atten(x)), inp)

        # 3. Only Wavelet branch (DWT -> Conv -> IDWT)
        def run_wavelet_only(x):
            curr_x_ll = x
            curr_shape = curr_x_ll.shape
            curr_x = op.wt_function(curr_x_ll)
            curr_x_ll = curr_x[:, :, 0, :, :]
            shape_x = curr_x.shape
            curr_x_tag = curr_x.reshape(shape_x[0], shape_x[1] * 4, shape_x[3], shape_x[4])
            curr_x_tag = op.wavelet_scale[0](op.wavelet_convs[0](curr_x_tag))
            curr_x_tag = curr_x_tag.reshape(shape_x)
            x_ll = curr_x_tag[:, :, 0, :, :]
            x_h = curr_x_tag[:, :, 1:4, :, :]
            curr_x = torch.cat([x_ll.unsqueeze(2), x_h], dim=2)
            next_x_ll = op.iwt_function(curr_x)
            return next_x_ll[:, :, :curr_shape[2], :curr_shape[3]]

        t_wavelet = time_op(run_wavelet_only, inp)

        print(f"\n--- {stage_name} ---")
        print(f"Total MBWTConv2d    : {t_total:.3f} ms")
        print(f"  Mamba SS2D Scan   : {t_mamba:.3f} ms ({100*t_mamba/t_total:.1f}%)")
        print(f"  Wavelet (DWT+IDWT): {t_wavelet:.3f} ms ({100*t_wavelet/t_total:.1f}%)")

print("=" * 80)
'''

result = subprocess.run(["/content/mobilemamba_env/bin/python", "-c", code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
# @title
#Test 4
import subprocess

code = r'''
import os
import sys
import copy
import time
import types
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ============================================================================
# ENVIRONMENT
# ============================================================================

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = "/content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth"
IMAGENET_VAL = "/content/imagenet/val"

sys.path.insert(0, REPO)
sys.path.insert(0, "/content/MobileMamba/model/lib_mamba/kernels/selective_scan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 105)
print("TESTING WAVELET BRANCH BYPASS (ZERO-SHOT)")
print("=" * 105)
print("Device     :", device)
print("Checkpoint :", CHECKPOINT_PATH)
print("ImageNet   :", IMAGENET_VAL)

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B4

def load_model():
    cfg = copy.deepcopy(CFG_MobileMamba_B4)
    model = MobileMamba(**cfg, num_classes=1000, distillation=False, forward_type="v052d")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
    state_dict = {(k.replace("module.", "", 1) if k.startswith("module.") else k): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    return model

# Bypassed forward pass for MBWTConv2d: skips DWT + Conv + IDWT completely
def bypassed_mbwt_forward(self, x):
    x = self.base_scale(self.global_atten(x))
    if self.do_stride is not None:
        x = self.do_stride(x)
    return x

def apply_wavelet_bypass(model, stages_to_bypass):
    # stages_to_bypass: list of stage indices, e.g. [1], [2], [1, 2, 3]
    count = 0
    for stage_idx in stages_to_bypass:
        blocks = getattr(model, f"blocks{stage_idx}")
        for block in blocks:
            if hasattr(block, "mixer") and hasattr(block.mixer, "m"):
                global_op = block.mixer.m.attn.global_op
                # Rebind forward method in memory
                global_op.forward = types.MethodType(bypassed_mbwt_forward, global_op)
                count += 1
    return model, count

# ============================================================================
# SMOKE TEST (FAIL-FAST)
# ============================================================================

print("\nRunning fail-fast smoke test on bypassed architecture...")
test_model = load_model()
test_model, patched_cnt = apply_wavelet_bypass(test_model, [1, 2, 3])
test_model = test_model.to(device).eval()

dummy = torch.randn(2, 3, 512, 512, device=device)
with torch.inference_mode():
    out = test_model(dummy)
    if isinstance(out, (tuple, list)):
        out = out[0]

assert out.shape == (2, 1000), f"Smoke test failed! Output shape: {out.shape}"
assert not torch.isnan(out).any(), "Smoke test failed! NaN output detected."
del test_model, dummy, out
torch.cuda.empty_cache()
print(f"Smoke test: PASS ({patched_cnt} MBWTConv2d blocks successfully bypassed)\n")

# ============================================================================
# DATASET SETUP (2048 IMAGES)
# ============================================================================

transform = transforms.Compose([
    transforms.Resize(585, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
subset = Subset(dataset, list(range(2048)))

accuracy_loader = DataLoader(
    subset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

@torch.inference_mode()
def evaluate_accuracy(model, name):
    model.eval()
    correct1, correct5, total = 0, 0, 0

    for images, labels in accuracy_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[0]

        _, pred = output.topk(5, dim=1, largest=True, sorted=True)
        correct = (pred == labels.unsqueeze(1))

        correct1 += correct[:, 0].sum().item()
        correct5 += correct.any(dim=1).sum().item()
        total += labels.size(0)

    top1 = 100.0 * correct1 / total
    top5 = 100.0 * correct5 / total
    print(f"{name} -> Top-1: {top1:.3f}% ({correct1}/{total}) | Top-5: {top5:.3f}% ({correct5}/{total})")
    return top1, top5

@torch.inference_mode()
def measure_latency(model):
    batch_size = 32
    warmup = 30
    iterations = 100

    dummy = torch.randn(batch_size, 3, 512, 512, device=device)
    model.eval()

    for _ in range(warmup):
        _ = model(dummy)
    torch.cuda.synchronize()

    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]

    for i in range(iterations):
        start_events[i].record()
        _ = model(dummy)
        end_events[i].record()

    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]

    times.sort()
    mean_ms = sum(times) / len(times)
    fps = batch_size / (mean_ms / 1000.0)

    del dummy
    return {"mean_ms": mean_ms, "fps": fps}

# ============================================================================
# RUN CONFIGURATIONS
# ============================================================================

experiments = [
    ("BASELINE", []),
    ("BYPASS WAVELET (STAGE 2)", [2]),
    ("BYPASS WAVELET (STAGE 1)", [1]),
    ("BYPASS WAVELET (ALL STAGES)", [1, 2, 3]),
]

results = {}

for name, bypass_stages in experiments:
    print("-" * 105)
    print(f"Benchmarking: {name}")
    print("-" * 105)

    model = load_model()
    if bypass_stages:
        model, count = apply_wavelet_bypass(model, bypass_stages)
        print(f"Patched {count} blocks across stages {bypass_stages}")

    model = model.to(device).eval()

    # Stage 1: Isolated latency on steady/cool GPU first
    latency = measure_latency(model)
    print(f"Mean Latency: {latency['mean_ms']:.3f} ms | FPS: {latency['fps']:.2f}")

    # Stage 2: Quick accuracy check
    top1, top5 = evaluate_accuracy(model, name)

    results[name] = {"top1": top1, "top5": top5, "latency": latency}

    del model
    torch.cuda.empty_cache()

# ============================================================================
# RESULTS COMPARISON
# ============================================================================

base = results["BASELINE"]
base_lat = base["latency"]["mean_ms"]
base_fps = base["latency"]["fps"]
base_top1 = base["top1"]
base_top5 = base["top5"]

print("\n" + "=" * 105)
print("FINAL EXPERIMENT RESULTS: WAVELET BRANCH OPTIMIZATION")
print("=" * 105)
header = f"{'Configuration':<32} {'Top-1':>8} {'ΔTop-1':>9} {'Top-5':>8} {'ΔTop-5':>9} {'Latency':>11} {'ΔLatency':>11} {'FPS':>8} {'ΔFPS':>8}"
print(header)
print("-" * 105)

for name in results:
    r = results[name]
    lat = r["latency"]["mean_ms"]
    fps = r["latency"]["fps"]

    top1_d = r["top1"] - base_top1
    top5_d = r["top5"] - base_top5
    lat_d = 100.0 * (lat - base_lat) / base_lat
    fps_d = 100.0 * (fps - base_fps) / base_fps

    print(
        f"{name:<32} "
        f"{r['top1']:>7.3f}% {top1_d:>+8.3f}% "
        f"{r['top5']:>7.3f}% {top5_d:>+8.3f}% "
        f"{lat:>8.3f} ms {lat_d:>+10.2f}% "
        f"{fps:>8.2f} {fps_d:>+7.2f}%"
    )

print("=" * 105)
'''

result = subprocess.run(["/content/mobilemamba_env/bin/python", "-c", code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
# @title
#Test 5
import subprocess

code = r'''
import os
import sys
import copy
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ============================================================================
# ENVIRONMENT
# ============================================================================

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = "/content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth"
IMAGENET_VAL = "/content/imagenet/val"

sys.path.insert(0, REPO)
sys.path.insert(0, "/content/MobileMamba/model/lib_mamba/kernels/selective_scan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 105)
print("STAGE 1 TOPOLOGY OPTIMIZATION (ADAPTING S6 DEPTH TO B4)")
print("=" * 105)
print("Device     :", device)
print("Checkpoint :", CHECKPOINT_PATH)
print("ImageNet   :", IMAGENET_VAL)

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B4

def load_model():
    cfg = copy.deepcopy(CFG_MobileMamba_B4)
    model = MobileMamba(**cfg, num_classes=1000, distillation=False, forward_type="v052d")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
    state_dict = {(k.replace("module.", "", 1) if k.startswith("module.") else k): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    return model

# ============================================================================
# SMOKE TESTS (FAIL-FAST)
# ============================================================================

print("\nRunning fail-fast smoke tests...")

# 1. Mixer-only bypass test
m1 = load_model()
m1.blocks1[1].mixer = nn.Identity()
m1 = m1.to(device).eval()

# 2. Full block bypass test
m2 = load_model()
m2.blocks1[1] = nn.Identity()
m2 = m2.to(device).eval()

dummy = torch.randn(2, 3, 512, 512, device=device)
with torch.inference_mode():
    out1 = m1(dummy)
    out2 = m2(dummy)
    if isinstance(out1, (tuple, list)): out1 = out1[0]
    if isinstance(out2, (tuple, list)): out2 = out2[0]

assert out1.shape == (2, 1000) and not torch.isnan(out1).any(), "Mixer bypass failed smoke test!"
assert out2.shape == (2, 1000) and not torch.isnan(out2).any(), "Full block bypass failed smoke test!"

del m1, m2, dummy, out1, out2
torch.cuda.empty_cache()
print("Smoke tests: PASS (Both modifications verified safe)\n")

# ============================================================================
# DATASET SETUP (2048 IMAGES)
# ============================================================================

transform = transforms.Compose([
    transforms.Resize(585, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
subset = Subset(dataset, list(range(2048)))

accuracy_loader = DataLoader(
    subset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

@torch.inference_mode()
def evaluate_accuracy(model, name):
    model.eval()
    correct1, correct5, total = 0, 0, 0

    for images, labels in accuracy_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[0]

        _, pred = output.topk(5, dim=1, largest=True, sorted=True)
        correct = (pred == labels.unsqueeze(1))

        correct1 += correct[:, 0].sum().item()
        correct5 += correct.any(dim=1).sum().item()
        total += labels.size(0)

    top1 = 100.0 * correct1 / total
    top5 = 100.0 * correct5 / total
    print(f"{name} -> Top-1: {top1:.3f}% ({correct1}/{total}) | Top-5: {top5:.3f}% ({correct5}/{total})")
    return top1, top5

@torch.inference_mode()
def measure_latency(model):
    batch_size = 32
    warmup = 30
    iterations = 100

    dummy = torch.randn(batch_size, 3, 512, 512, device=device)
    model.eval()

    for _ in range(warmup):
        _ = model(dummy)
    torch.cuda.synchronize()

    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]

    for i in range(iterations):
        start_events[i].record()
        _ = model(dummy)
        end_events[i].record()

    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]

    times.sort()
    mean_ms = sum(times) / len(times)
    fps = batch_size / (mean_ms / 1000.0)

    del dummy
    return {"mean_ms": mean_ms, "fps": fps}

# ============================================================================
# EXPERIMENT EXECUTION
# ============================================================================

experiments = [
    ("RAW BASELINE", "none"),
    ("STAGE 1: MIXER-1 BYPASS", "mixer"),
    ("STAGE 1: FULL BLOCK-1 BYPASS (S6 DEPTH)", "block"),
]

results = {}

for name, mode in experiments:
    print("-" * 105)
    print(f"Benchmarking: {name}")
    print("-" * 105)

    model = load_model()
    if mode == "mixer":
        model.blocks1[1].mixer = nn.Identity()
    elif mode == "block":
        model.blocks1[1] = nn.Identity()

    model = model.to(device).eval()

    # Stage 1: Isolated latency on steady/cool GPU first
    latency = measure_latency(model)
    print(f"Mean Latency: {latency['mean_ms']:.3f} ms | FPS: {latency['fps']:.2f}")

    # Stage 2: Quick accuracy check
    top1, top5 = evaluate_accuracy(model, name)

    results[name] = {"top1": top1, "top5": top5, "latency": latency}

    del model
    torch.cuda.empty_cache()

# ============================================================================
# RESULTS SUMMARY
# ============================================================================

base = results["RAW BASELINE"]
base_lat = base["latency"]["mean_ms"]
base_fps = base["latency"]["fps"]
base_top1 = base["top1"]
base_top5 = base["top5"]

print("\n" + "=" * 105)
print("FINAL EXPERIMENT RESULTS: STAGE 1 TOPOLOGY")
print("=" * 105)
header = f"{'Configuration':<40} {'Top-1':>8} {'ΔTop-1':>9} {'Top-5':>8} {'ΔTop-5':>9} {'Latency':>11} {'ΔLatency':>11} {'FPS':>8} {'ΔFPS':>8}"
print(header)
print("-" * 105)

for name in results:
    r = results[name]
    lat = r["latency"]["mean_ms"]
    fps = r["latency"]["fps"]

    top1_d = r["top1"] - base_top1
    top5_d = r["top5"] - base_top5
    lat_d = 100.0 * (lat - base_lat) / base_lat
    fps_d = 100.0 * (fps - base_fps) / base_fps

    print(
        f"{name:<40} "
        f"{r['top1']:>7.3f}% {top1_d:>+8.3f}% "
        f"{r['top5']:>7.3f}% {top5_d:>+8.3f}% "
        f"{lat:>8.3f} ms {lat_d:>+10.2f}% "
        f"{fps:>8.2f} {fps_d:>+7.2f}%"
    )

print("=" * 105)
'''

result = subprocess.run(["/content/mobilemamba_env/bin/python", "-c", code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
# @title
#Test 6
import subprocess

code = r'''
import os
import sys
import copy
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ============================================================================
# ENVIRONMENT
# ============================================================================

REPO = "/content/MobileMamba"
CHECKPOINT_PATH = "/content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth"
IMAGENET_VAL = "/content/imagenet/val"

sys.path.insert(0, REPO)
sys.path.insert(0, "/content/MobileMamba/model/lib_mamba/kernels/selective_scan")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 105)
print("EXPERIMENT: S6 DEPTH TOPOLOGY [1, 2, 2] APPLIED TO MOBILEMAMBA-B4")
print("=" * 105)
print("Device     :", device)
print("Checkpoint :", CHECKPOINT_PATH)
print("ImageNet   :", IMAGENET_VAL)

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B4

def load_model():
    cfg = copy.deepcopy(CFG_MobileMamba_B4)
    model = MobileMamba(**cfg, num_classes=1000, distillation=False, forward_type="v052d")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
    state_dict = {(k.replace("module.", "", 1) if k.startswith("module.") else k): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    return model

# ============================================================================
# SMOKE TESTS (FAIL-FAST)
# ============================================================================

print("\nRunning fail-fast smoke tests...")
test_model = load_model()
test_model.blocks1[1] = nn.Identity()
test_model.blocks2[5] = nn.Identity()
test_model = test_model.to(device).eval()

dummy = torch.randn(2, 3, 512, 512, device=device)
with torch.inference_mode():
    out = test_model(dummy)
    if isinstance(out, (tuple, list)):
        out = out[0]

assert out.shape == (2, 1000), f"Shape mismatch: {out.shape}"
assert not torch.isnan(out).any(), "NaN detected in smoke test output!"
del test_model, dummy, out
torch.cuda.empty_cache()
print("Smoke tests: PASS (Topology [1, 2, 2] verified safe)\n")

# ============================================================================
# DATASET SETUP (2048 IMAGES)
# ============================================================================

transform = transforms.Compose([
    transforms.Resize(585, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(IMAGENET_VAL, transform=transform)
subset = Subset(dataset, list(range(2048)))

accuracy_loader = DataLoader(
    subset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

@torch.inference_mode()
def evaluate_accuracy(model, name):
    model.eval()
    correct1, correct5, total = 0, 0, 0

    for images, labels in accuracy_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[0]

        _, pred = output.topk(5, dim=1, largest=True, sorted=True)
        correct = (pred == labels.unsqueeze(1))

        correct1 += correct[:, 0].sum().item()
        correct5 += correct.any(dim=1).sum().item()
        total += labels.size(0)

    top1 = 100.0 * correct1 / total
    top5 = 100.0 * correct5 / total
    print(f"{name} -> Top-1: {top1:.3f}% ({correct1}/{total}) | Top-5: {top5:.3f}% ({correct5}/{total})")
    return top1, top5

@torch.inference_mode()
def measure_latency(model):
    batch_size = 32
    warmup = 30
    iterations = 100

    dummy = torch.randn(batch_size, 3, 512, 512, device=device)
    model.eval()

    for _ in range(warmup):
        _ = model(dummy)
    torch.cuda.synchronize()

    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(iterations)]

    for i in range(iterations):
        start_events[i].record()
        _ = model(dummy)
        end_events[i].record()

    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]

    times.sort()
    mean_ms = sum(times) / len(times)
    fps = batch_size / (mean_ms / 1000.0)

    del dummy
    return {"mean_ms": mean_ms, "fps": fps}

# ============================================================================
# EXPERIMENT EXECUTION
# ============================================================================

experiments = [
    ("RAW BASELINE (DEPTH [2, 3, 2])", False, False),
    ("STAGE 1 BLOCK-1 BYPASS ([1, 3, 2])", True, False),
    ("STAGE 2 BLOCK-5 BYPASS ([2, 2, 2])", False, True),
    ("S6 TOPOLOGY ON B4 ([1, 2, 2])", True, True),
]

results = {}

for name, bypass_s1, bypass_s2 in experiments:
    print("-" * 105)
    print(f"Benchmarking: {name}")
    print("-" * 105)

    model = load_model()
    if bypass_s1:
        model.blocks1[1] = nn.Identity()
    if bypass_s2:
        model.blocks2[5] = nn.Identity()

    model = model.to(device).eval()

    # Stage 1: Isolated latency on steady/cool GPU first
    latency = measure_latency(model)
    print(f"Mean Latency: {latency['mean_ms']:.3f} ms | FPS: {latency['fps']:.2f}")

    # Stage 2: Quick accuracy check
    top1, top5 = evaluate_accuracy(model, name)

    results[name] = {"top1": top1, "top5": top5, "latency": latency}

    del model
    torch.cuda.empty_cache()

# ============================================================================
# RESULTS SUMMARY
# ============================================================================

base = results["RAW BASELINE (DEPTH [2, 3, 2])"]
base_lat = base["latency"]["mean_ms"]
base_fps = base["latency"]["fps"]
base_top1 = base["top1"]
base_top5 = base["top5"]

print("\n" + "=" * 105)
print("FINAL RESULTS: S6 TOPOLOGY EXPLORATION ON B4")
print("=" * 105)
header = f"{'Configuration':<40} {'Top-1':>8} {'ΔTop-1':>9} {'Top-5':>8} {'ΔTop-5':>9} {'Latency':>11} {'ΔLatency':>11} {'FPS':>8} {'ΔFPS':>8}"
print(header)
print("-" * 105)

for name in results:
    r = results[name]
    lat = r["latency"]["mean_ms"]
    fps = r["latency"]["fps"]

    top1_d = r["top1"] - base_top1
    top5_d = r["top5"] - base_top5
    lat_d = 100.0 * (lat - base_lat) / base_lat
    fps_d = 100.0 * (fps - base_fps) / base_fps

    print(
        f"{name:<40} "
        f"{r['top1']:>7.3f}% {top1_d:>+8.3f}% "
        f"{r['top5']:>7.3f}% {top5_d:>+8.3f}% "
        f"{lat:>8.3f} ms {lat_d:>+10.2f}% "
        f"{fps:>8.2f} {fps_d:>+7.2f}%"
    )

print("=" * 105)
'''

result = subprocess.run(["/content/mobilemamba_env/bin/python", "-c", code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

# **FULL EVAL**

**Baseline**

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B1 Baseline
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_baseline.py \
  --variant b1 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B1/mobilemamba_b1.pth \
  --save-path /content/B1_ImageNet_baseline_PROPER.json

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B2 Baseline
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_baseline.py \
  --variant b2 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B2/mobilemamba_b2.pth \
  --save-path /content/B2_ImageNet_baseline_PROPER.json

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B4 Baseline
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_baseline.py \
  --variant b4 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth \
  --save-path /content/B4_ImageNet_baseline_PROPER.json

**Path-A**

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B1 Path A
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_path_a.py \
  --variant b1 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B1/mobilemamba_b1.pth \
  --save-path /content/B1_ImageNet_PathA_PROPER.json

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B2 Path A
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_path_a.py \
  --variant b2 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B2/mobilemamba_b2.pth \
  --save-path /content/B2_ImageNet_PathA_PROPER.json

In [ ]:
# @title Run Full 50k ImageNet Benchmark on MobileMamba-B4 Path A
%cd /content/MobileMamba

!LD_LIBRARY_PATH=/usr/lib64-nvidia:/content/cuda-11.8/lib64:$LD_LIBRARY_PATH \
CUDA_HOME=/content/cuda-11.8 \
PATH=/content/cuda-11.8/bin:$PATH \
PYTHONPATH=/content/MobileMamba/model/lib_mamba/kernels/selective_scan:/content/MobileMamba:$PYTHONPATH \
/content/mobilemamba_env/bin/python /content/benchmark_path_a.py \
  --variant b4 \
  --checkpoint /content/MobileMamba/weights/MobileMamba_B4/mobilemamba_b4.pth \
  --save-path /content/B4_ImageNet_PathA_PROPER.json